# Chapter 5-4 [해답]. 백테스팅 실습과제 풀이

> **KDT AI 퀀트 · 백테스트/성과검증 파트 — 실습과제 완성 코드**

이 노트북은 **Chapter 5-4 본편**의 실습과제에 대한 **참고 해답**입니다.
- **Part A. 응용 실습(본편 3장)** — 파라미터 민감도 / 손절 / 거래비용 / 멀티종목
- **Part B. 심화 실습(본편 6장)** — Anchored 워크포워드 / IS·OOS 길이 민감도 / 파라미터 안정성 히트맵 / 거래비용 반영 워크포워드 / 레짐 필터 효과 분리

> ⚠️ 정답은 하나가 아닙니다. 아래 코드는 **"이렇게 접근한다"는 예시**이며,
> 핵심은 **결과를 해석**하는 것입니다. 먼저 스스로 풀어본 뒤 대조해 보세요.


---
## 0. 환경 설정 & 핵심 함수 재정의

해답 노트북만 단독 실행해도 되도록, 본편에서 만든 함수를 그대로 다시 정의합니다.
> 단, 뒤 실습(레짐 필터 효과)을 위해 `run_backtest` 에 `use_regime` 옵션 하나를 추가했습니다.
> (기본값 `True` → 본편과 완전히 동일하게 동작)


In [ ]:
!pip install -q finance-datareader

import FinanceDataReader as fdr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['axes.grid'] = True


In [ ]:
# ---- 지표 ----
def compute_rsi(close, period=14):
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    return 100 - 100 / (1 + avg_gain / avg_loss)

# ---- 백테스트 (use_regime 옵션 추가) ----
def run_backtest(df, trend_window=200, rsi_period=14, rsi_entry=45, rsi_exit=80,
                 slippage=0.004, init_cash=1_000_000, start=None, end=None,
                 use_regime=True):
    data = df.copy()
    data['trend_ma'] = data['close'].rolling(trend_window).mean()
    data['rsi'] = compute_rsi(data['close'], rsi_period)
    data['rsi_prev'] = data['rsi'].shift(1)
    sim = data.loc[start:end]

    cash, shares = float(init_cash), 0.0
    equity = []
    for _, row in sim.iterrows():
        price = row['close']
        ready = not (np.isnan(row['rsi']) or np.isnan(row['rsi_prev']) or
                     (use_regime and np.isnan(row['trend_ma'])))
        if ready:
            crossed_up = (row['rsi_prev'] < rsi_entry) and (row['rsi'] >= rsi_entry)
            regime_ok = (not use_regime) or (price > row['trend_ma'])
            if shares == 0 and regime_ok and crossed_up:
                shares = cash / (price * (1 + slippage)); cash = 0.0
            elif shares > 0 and row['rsi'] >= rsi_exit:
                cash = shares * price * (1 - slippage); shares = 0.0
        equity.append(cash + shares * price)
    return pd.Series(equity, index=sim.index, name='equity')

# ---- 성과지표 ----
def performance(equity, periods_per_year=250, rf=0.0):
    equity = pd.Series(equity).reset_index(drop=True).astype(float)
    total_return = equity.iloc[-1] / equity.iloc[0] - 1
    years = len(equity) / periods_per_year
    cagr = (equity.iloc[-1] / equity.iloc[0]) ** (1 / years) - 1 if years > 0 else np.nan
    daily = equity.pct_change().dropna()
    sharpe = ((daily - rf/periods_per_year).mean() / daily.std()) * np.sqrt(periods_per_year) if daily.std() > 0 else np.nan
    dd = equity / equity.cummax() - 1
    return {'total_return': total_return, 'cagr': cagr, 'sharpe': sharpe, 'mdd': dd.min()}

def summary(name, equity):
    m = performance(equity)
    print(f"[{name}]  총수익 {m['total_return']*100:7.2f}% | CAGR {m['cagr']*100:6.2f}% | "
          f"Sharpe {m['sharpe']:5.2f} | MDD {m['mdd']*100:7.2f}%")
    return m

# ---- 그리드서치 / 워크포워드 (use_regime 전달) ----
def grid_search(df, period_list, entry_list, exit_list, start, end,
                trend_window=200, metric='sharpe', use_regime=True):
    rows = []
    for p, en, ex in itertools.product(period_list, entry_list, exit_list):
        eq = run_backtest(df, trend_window=trend_window, rsi_period=p, rsi_entry=en,
                          rsi_exit=ex, start=start, end=end, use_regime=use_regime)
        rows.append({'rsi_p': p, 'entry': en, 'exit': ex, **performance(eq)})
    return pd.DataFrame(rows).sort_values(metric, ascending=False).reset_index(drop=True)

def walk_forward(df, period_list, entry_list, exit_list, is_len=500, oos_len=125,
                 trend_window=200, metric='sharpe', init_cash=1_000_000, use_regime=True):
    idx = df.index; n = len(df)
    segments, log = [], []
    i = 0; cash = init_cash
    while i + is_len + oos_len <= n:
        is_start, is_end = idx[i], idx[i + is_len - 1]
        oos_start, oos_end = idx[i + is_len], idx[i + is_len + oos_len - 1]
        res = grid_search(df, period_list, entry_list, exit_list, is_start, is_end,
                          trend_window, metric, use_regime)
        b = res.iloc[0]
        eq = run_backtest(df, trend_window=trend_window, rsi_period=int(b.rsi_p),
                          rsi_entry=int(b.entry), rsi_exit=int(b['exit']), init_cash=cash,
                          start=oos_start, end=oos_end, use_regime=use_regime)
        cash = eq.iloc[-1]; segments.append(eq); m = performance(eq)
        log.append({'oos_start': oos_start.date(), 'oos_end': oos_end.date(),
                    'rsi_p': int(b.rsi_p), 'entry': int(b.entry), 'exit': int(b['exit']),
                    'oos_cagr': m['cagr'], 'oos_sharpe': m['sharpe'], 'oos_mdd': m['mdd']})
        i += oos_len
    return pd.concat(segments), pd.DataFrame(log)


In [ ]:
# 데이터
df = fdr.DataReader('005930', '2015').rename(columns=str.lower)[['close']].copy()

# 공통 그리드
PERIOD_LIST = [7, 14, 21]
ENTRY_LIST  = [40, 45, 50]
EXIT_LIST   = [65, 70, 75, 80]

print(f"기간: {df.index[0].date()} ~ {df.index[-1].date()} ({len(df)} 일)")
summary('기본 전략(RSI45/80)', run_backtest(df))


---
# Part A. 응용 실습(본편 3장) 해답


## 실습 3-1. RSI 임계값 민감도 분석

`rsi_exit` 뿐 아니라 `rsi_entry`, `rsi_period` 까지 한 번에 훑어 **"안정 구간"** 을 찾습니다.


In [ ]:
# (1) 청산 임계값(rsi_exit) 민감도
print("● rsi_exit 민감도")
rows = [{'rsi_exit': ex, **performance(run_backtest(df, rsi_exit=ex))}
        for ex in [65, 70, 75, 80, 85]]
display(pd.DataFrame(rows).set_index('rsi_exit').round(3))

# (2) 진입 임계값(rsi_entry) 민감도  ← 30이면 거의 거래가 없음을 확인
print("\n● rsi_entry 민감도 (30은 진입이 거의 없어 성과 밋밋)")
rows = [{'rsi_entry': en, **performance(run_backtest(df, rsi_entry=en))}
        for en in [30, 35, 40, 45, 50, 55]]
display(pd.DataFrame(rows).set_index('rsi_entry').round(3))

# (3) 2D: entry x exit 의 CAGR 표
print("\n● entry × exit → CAGR(%) 표")
tbl = pd.DataFrame(index=[35,40,45,50], columns=[65,70,75,80], dtype=float)
for en in tbl.index:
    for ex in tbl.columns:
        tbl.loc[en, ex] = performance(run_backtest(df, rsi_entry=en, rsi_exit=ex))['cagr'] * 100
display(tbl.round(2))
print("해석: 특정 한 칸만 튀는 게 아니라 '넓은 구간'에서 고르게 양(+)이면 견고한 전략.")


## 실습 3-2. 손절(Stop-Loss) 추가

진입가 대비 `stop_loss`(예: 7%) 이상 하락하면 **RSI 80을 기다리지 않고 즉시 매도**합니다.


In [ ]:
def run_backtest_sl(df, trend_window=200, rsi_period=14, rsi_entry=45, rsi_exit=80,
                    stop_loss=0.07, slippage=0.004, init_cash=1_000_000,
                    start=None, end=None):
    data = df.copy()
    data['trend_ma'] = data['close'].rolling(trend_window).mean()
    data['rsi'] = compute_rsi(data['close'], rsi_period)
    data['rsi_prev'] = data['rsi'].shift(1)
    sim = data.loc[start:end]

    cash, shares, entry_price = float(init_cash), 0.0, 0.0
    equity = []
    for _, row in sim.iterrows():
        price = row['close']
        ready = not (np.isnan(row['trend_ma']) or np.isnan(row['rsi']) or np.isnan(row['rsi_prev']))
        if ready:
            crossed_up = (row['rsi_prev'] < rsi_entry) and (row['rsi'] >= rsi_entry)
            if shares == 0 and price > row['trend_ma'] and crossed_up:
                shares = cash / (price * (1 + slippage)); cash = 0.0; entry_price = price
            # ★ 손절: 진입가 대비 stop_loss 이상 하락하면 즉시 매도
            elif shares > 0 and price <= entry_price * (1 - stop_loss):
                cash = shares * price * (1 - slippage); shares = 0.0
            # 정상 청산: RSI 80
            elif shares > 0 and row['rsi'] >= rsi_exit:
                cash = shares * price * (1 - slippage); shares = 0.0
        equity.append(cash + shares * price)
    return pd.Series(equity, index=sim.index, name='equity')


print("손절 유무 비교")
summary('손절 없음        ', run_backtest(df))
for sl in [0.05, 0.07, 0.10]:
    summary(f'손절 {int(sl*100)}%        ', run_backtest_sl(df, stop_loss=sl))
print("\n관찰: 손절은 대개 MDD를 줄이지만, 너무 타이트하면 반등 전에 털려 수익도 깎일 수 있음.")


## 실습 3-3. 거래비용 민감도

슬리피지를 키우며 CAGR이 얼마나 깎이는지 + **연간 거래 횟수**까지 함께 봅니다.
(거래가 잦을수록 비용에 민감)


In [ ]:
def count_trades(df, **kw):
    """매수 체결 횟수를 세는 간이 버전."""
    data = df.copy()
    data['trend_ma'] = data['close'].rolling(kw.get('trend_window', 200)).mean()
    data['rsi'] = compute_rsi(data['close'], kw.get('rsi_period', 14))
    data['rsi_prev'] = data['rsi'].shift(1)
    en, ex = kw.get('rsi_entry', 45), kw.get('rsi_exit', 80)
    shares, buys = 0, 0
    for _, r in data.iterrows():
        if np.isnan(r['trend_ma']) or np.isnan(r['rsi_prev']): continue
        if shares == 0 and r['close'] > r['trend_ma'] and r['rsi_prev'] < en and r['rsi'] >= en:
            shares = 1; buys += 1
        elif shares > 0 and r['rsi'] >= ex:
            shares = 0
    return buys

n_buys = count_trades(df)
years = len(df) / 250
print(f"총 매수 {n_buys}회 (연 평균 {n_buys/years:.1f}회)\n")
for s in [0.0, 0.002, 0.004, 0.01]:
    m = performance(run_backtest(df, slippage=s))
    print(f"slippage {s*100:4.1f}% -> CAGR {m['cagr']*100:6.2f}% | Sharpe {m['sharpe']:.2f}")
print("\n해석: 이 전략은 거래가 드물어 비용 민감도가 낮은 편. "
      "고빈도 전략이라면 같은 슬리피지에도 CAGR이 크게 무너진다.")


## 실습 3-4. 멀티 종목으로 확장 (동일가중 포트폴리오)

여러 종목에 자본을 나눠 투자하고 합산해 **분산 효과(MDD 감소)** 를 확인합니다.


In [ ]:
tickers = ['005930', '000660', '005380', '035420', '051910']  # 삼성전자, SK하이닉스, 현대차, NAVER, LG화학
cap_each = 1_000_000

curves = {}
for t in tickers:
    d = fdr.DataReader(t, '2015').rename(columns=str.lower)[['close']]
    curves[t] = run_backtest(d, init_cash=cap_each)

# 날짜 정렬 후 동일가중 합산 (각 종목 100만원씩 → 총 500만원 투입)
port = pd.concat(curves, axis=1)
port.columns = tickers
port = port.ffill().dropna()
port_total = port.sum(axis=1)

print("● 개별 종목 성과")
for t in tickers:
    summary(t, port[t])
print("\n● 동일가중 포트폴리오 (5종목)")
summary('Portfolio', port_total)

plt.figure(figsize=(14, 6))
for t in tickers:
    (port[t] / port[t].iloc[0]).plot(alpha=0.5, label=t)
(port_total / port_total.iloc[0]).plot(color='k', lw=2.5, label='Portfolio (equal weight)')
plt.title('Multi-Stock RSI Strategy: Individual vs Equal-Weight Portfolio')
plt.ylabel('Growth (x)'); plt.legend(); plt.show()
print("해석: 개별 종목 대비 포트폴리오의 MDD가 완만해지면 분산 효과가 작동한 것.")


---
# Part B. 심화 실습(본편 6장) 해답


## 심화 1. Anchored(고정 시작점) 워크포워드

본편 `walk_forward` 는 IS 창이 앞으로 **밀리는(Rolling)** 방식입니다.
Anchored는 IS **시작점을 고정**하고 창을 계속 **늘려**갑니다 (데이터가 쌓일수록 더 많은 과거로 학습).


In [ ]:
def walk_forward_anchored(df, period_list, entry_list, exit_list,
                          is_len=500, oos_len=125, trend_window=200,
                          metric='sharpe', init_cash=1_000_000, use_regime=True):
    """Anchored: IS 시작은 항상 맨 앞(idx[0]), IS 끝만 뒤로 확장."""
    idx = df.index; n = len(df)
    segments, log = [], []
    i = 0; cash = init_cash
    while i + is_len + oos_len <= n:
        is_start = idx[0]                       # ★ 시작점 고정
        is_end   = idx[i + is_len - 1]          # ★ 끝점만 확장
        oos_start, oos_end = idx[i + is_len], idx[i + is_len + oos_len - 1]
        b = grid_search(df, period_list, entry_list, exit_list, is_start, is_end,
                        trend_window, metric, use_regime).iloc[0]
        eq = run_backtest(df, trend_window=trend_window, rsi_period=int(b.rsi_p),
                          rsi_entry=int(b.entry), rsi_exit=int(b['exit']), init_cash=cash,
                          start=oos_start, end=oos_end, use_regime=use_regime)
        cash = eq.iloc[-1]; segments.append(eq); m = performance(eq)
        log.append({'oos_start': oos_start.date(), 'entry': int(b.entry),
                    'exit': int(b['exit']), 'oos_sharpe': m['sharpe']})
        i += oos_len
    return pd.concat(segments), pd.DataFrame(log)


wf_roll,   _ = walk_forward(df, PERIOD_LIST, ENTRY_LIST, EXIT_LIST, is_len=500, oos_len=125)
wf_anchor, _ = walk_forward_anchored(df, PERIOD_LIST, ENTRY_LIST, EXIT_LIST, is_len=500, oos_len=125)

summary('Rolling  WF', wf_roll)
summary('Anchored WF', wf_anchor)

plt.figure(figsize=(14, 6))
(wf_roll / wf_roll.iloc[0]).plot(label='Rolling', color='k')
(wf_anchor / wf_anchor.iloc[0]).plot(label='Anchored', color='teal')
plt.title('Rolling vs Anchored Walk-Forward (OOS)'); plt.ylabel('Growth (x)')
plt.legend(); plt.show()
print("해석: Anchored는 과거 전체를 쓰므로 파라미터가 더 안정적이나, 옛 국면(regime)에 끌릴 수 있음.")


## 심화 2. `is_len` / `oos_len` 민감도

워크포워드 결과가 **설정값에 얼마나 민감한지** 표로 정리합니다.
> 한 설정에서만 좋다면 그 자체가 위험 신호(설정에 과최적화).


In [ ]:
# ⏱️ 이 셀은 여러 번의 워크포워드를 돌리므로 가장 무겁습니다(수십 초~수 분).
#    속도를 위해 탐색 그리드를 가볍게 줄여서 사용합니다.
LP, LE, LX = [14], [40, 45, 50], [70, 75, 80]   # 가벼운 그리드

rows = []
for is_len in [250, 500, 750]:
    for oos_len in [63, 125, 250]:
        wf, log = walk_forward(df, LP, LE, LX, is_len=is_len, oos_len=oos_len)
        m = performance(wf)
        rows.append({'is_len': is_len, 'oos_len': oos_len, 'folds': len(log),
                     'CAGR%': m['cagr']*100, 'Sharpe': m['sharpe'], 'MDD%': m['mdd']*100})

sens = pd.DataFrame(rows)
display(sens.round(2))

# Sharpe 를 피벗으로 한눈에
print("\nSharpe 피벗 (행=is_len, 열=oos_len)")
display(sens.pivot(index='is_len', columns='oos_len', values='Sharpe').round(2))
print("해석: 값들이 대체로 고르게 양(+)이면 견고. 특정 칸만 튀면 그 설정에 운좋게 맞은 것.")


## 심화 3. 파라미터 안정성 히트맵

특정 IS 구간에서 (entry × exit) 조합별 Sharpe를 2D 히트맵으로 그립니다.
**최적점이 "뾰족한 봉우리"인지 "넓은 고원"인지** 가 신뢰도의 핵심 — 고원일수록 견고합니다.


In [ ]:
# 첫 IS 구간(앞 500일)에서 entry × exit 격자의 Sharpe
is_end = df.index[500 - 1]
entries = [35, 40, 45, 50, 55]
exits   = [60, 65, 70, 75, 80, 85]

H = np.full((len(entries), len(exits)), np.nan)
for i, en in enumerate(entries):
    for j, ex in enumerate(exits):
        eq = run_backtest(df, rsi_entry=en, rsi_exit=ex, end=is_end)
        H[i, j] = performance(eq)['sharpe']

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(H, cmap='RdYlGn', aspect='auto', origin='lower')
ax.set_xticks(range(len(exits)));   ax.set_xticklabels(exits)
ax.set_yticks(range(len(entries))); ax.set_yticklabels(entries)
ax.set_xlabel('rsi_exit'); ax.set_ylabel('rsi_entry')
ax.set_title('Sharpe Heatmap over (entry x exit)  [IS window]')
for i in range(len(entries)):
    for j in range(len(exits)):
        ax.text(j, i, f'{H[i,j]:.2f}', ha='center', va='center', fontsize=9)
fig.colorbar(im, label='Sharpe'); plt.show()
print("해석: 초록(고Sharpe)이 넓게 이어진 '고원'이면 신뢰 가능. "
      "한 칸만 초록이고 주변이 빨강이면 그 값은 우연일 가능성이 큼.")


## 심화 4. 거래비용을 반영한 워크포워드

슬리피지를 현실적으로 높였을 때도 워크포워드 OOS가 **벤치마크(Buy & Hold)** 를 이기는지 검증합니다.
> `walk_forward` 는 내부에서 `run_backtest`(기본 slippage=0.004)를 호출하므로,
> 비용 비교를 위해 슬리피지를 주입할 수 있는 래퍼를 만들어 비교합니다.


In [ ]:
import functools

def walk_forward_cost(df, slippage, **kw):
    """run_backtest의 기본 slippage를 바꿔 끼운 walk_forward."""
    orig = globals()['run_backtest']
    patched = functools.partial(orig, slippage=slippage)
    globals()['run_backtest'] = patched
    try:
        return walk_forward(df, PERIOD_LIST, ENTRY_LIST, EXIT_LIST, **kw)
    finally:
        globals()['run_backtest'] = orig   # 원복

# 벤치마크: 워크포워드 OOS 구간에 맞춘 Buy & Hold
wf_ref, _ = walk_forward(df, PERIOD_LIST, ENTRY_LIST, EXIT_LIST)
bh = df['close'].loc[wf_ref.index[0]:]
bh = bh / bh.iloc[0] * 1_000_000

print("슬리피지별 워크포워드 OOS 성과 (vs Buy&Hold)")
summary('Buy & Hold      ', bh)
print('-'*70)
for s in [0.0, 0.004, 0.01, 0.02]:
    wf, _ = walk_forward_cost(df, slippage=s)
    summary(f'WF slippage {s*100:4.1f}%', wf)
print("\n해석: 슬리피지를 높일수록 OOS 성과가 깎인다. "
      "어느 비용 수준까지 벤치마크를 이기는지가 '실전 배포 가능성'의 기준.")


## 심화 5. 레짐 필터(200일선) 효과 분리

같은 RSI 전략을 **필터 ON vs OFF** 로 돌려, 200일선 필터가 실제로 **손실(MDD)을 줄이는지** 확인합니다.
> `use_regime=False` 로 두면 `price > 200일선` 조건을 무시하고 RSI 신호만으로 매매합니다.


In [ ]:
# (1) 전체 기간 단순 비교
print("● 전체 기간")
summary('필터 ON  (200일선)', run_backtest(df, use_regime=True))
summary('필터 OFF (RSI만)  ', run_backtest(df, use_regime=False))

# (2) 워크포워드 OOS 비교 (더 공정)
print("\n● 워크포워드 OOS")
wf_on,  _ = walk_forward(df, PERIOD_LIST, ENTRY_LIST, EXIT_LIST, use_regime=True)
wf_off, _ = walk_forward(df, PERIOD_LIST, ENTRY_LIST, EXIT_LIST, use_regime=False)
summary('필터 ON  WF', wf_on)
summary('필터 OFF WF', wf_off)

plt.figure(figsize=(14, 6))
(wf_on  / wf_on.iloc[0]).plot(label='Regime ON (MA200)', color='teal', lw=2)
(wf_off / wf_off.iloc[0]).plot(label='Regime OFF (RSI only)', color='red', alpha=0.7)
plt.title('Effect of the 200-day Trend Filter (Walk-Forward OOS)')
plt.ylabel('Growth (x)'); plt.legend(); plt.show()
print("해석: 필터 OFF는 하락장에서도 RSI 신호로 진입해 물릴 수 있어 보통 MDD가 커진다. "
      "필터 ON이 MDD를 줄여주면, 200일선 레짐 필터가 '방어' 역할을 한다는 증거.")


---
## 마무리

- 실습의 목적은 **"정답 숫자"가 아니라 방법론**입니다: 파라미터를 넓게 훑고, OOS/워크포워드로 검증하고, 비용·필터의 효과를 **수치로** 확인하는 습관.
- 좋은 전략의 3조건을 다시 기억하세요: **① OOS 생존 ② 비용 후 초과수익 ③ 파라미터 안정성.**

> 여기 코드는 한 가지 접근일 뿐입니다. 진입/청산 로직, 종목 유니버스, 리밸런싱 주기 등을 바꿔
> **자신만의 전략 파이프라인**으로 확장해 보세요.
